[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/59_triangulate_polygon.ipynb)

# 🔴 Hard: Polygon Triangulation (Ear-Clipping)

Given a simple (non-self-intersecting) convex **or concave** polygon with vertices in counter-clockwise order, decompose it into triangles using the **ear-clipping** algorithm. Return the indices of each triangle's vertices.

An **ear** is a vertex `B` (with polygon neighbours `A` and `C`) such that:
1. The turn `A → B → C` is **convex** (left turn): `cross(B−A, C−A) > 0`
2. **No other active vertex** lies inside or on the boundary of triangle `ABC`

Clip one ear per iteration (recording the triangle `(A, B, C)` and removing `B`), repeat until 3 vertices remain.

### Signature
```python
def triangulate_polygon(vertices: np.ndarray) -> np.ndarray:
    # vertices: (N, 2) float — simple polygon, CCW vertex order
    # returns:  (N-2, 3) int — each row is three vertex indices (into `vertices`)
```

### Rules
- Do **NOT** use a Python `for` loop over the `K` active vertices to check point-in-triangle — vectorise it
- You may use a `while` loop over triangulation steps (unavoidable: sequential ear removal)
- Use `>= 0` (not strict `> 0`) for the point-in-triangle test so that vertices landing exactly on an ear's edge are caught; exclude the ear's own three vertices with a boolean mask

### Example
```
# L-shape (CCW, 6 vertices)
vertices = [[0,0],[4,0],[4,2],[2,2],[2,4],[0,4]]
#         vertex 3 at (2,2) is reflex — cannot be an ear
output: [[0,1,5],[1,4,5],[1,2,4],[2,3,4]]  # one valid decomposition
#        (other orderings also correct — just verify total area == 12)
```

> **Reduction step (say this before coding):** at each step, vertex `B` is an ear iff `cross(B−A, C−A) > 0` AND for every other active vertex `P`, NOT all three of `cross(AB, AP) >= 0`, `cross(BC, BP) >= 0`, `cross(CA, CP) >= 0` hold simultaneously — test all `K` candidate ears against all `K` points in a single `(K, K)` broadcast, then exclude A/B/C of each ear via a boolean mask before calling `.any(axis=0)`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def triangulate_polygon(vertices):
    # vertices: (N, 2) — simple polygon, CCW
    # returns:  (N-2, 3) int — triangle vertex indices
    #
    # Sketch:
    # idx = np.arange(N)          # active vertex indices
    # while len(idx) > 3:
    #     K = len(idx)
    #     v = vertices[idx]        # (K, 2)
    #     A = v[(arange-1)%K]      # (K, 2) prev neighbours
    #     B = v                    # (K, 2)
    #     C = v[(arange+1)%K]      # (K, 2) next neighbours
    #
    #     convex = cross(B-A, C-A) > 0              # (K,)
    #
    #     # Vectorised point-in-triangle (>= 0, not strict):
    #     P  = v[:, None, :]   # (K, 1, 2)
    #     Aj = A[None, :, :]   # (1, K, 2)  ← ear candidates
    #     ...compute (K,K) inside matrix with >= 0...
    #
    #     # Exclude A, B, C of each ear j via boolean mask
    #     exclude[(j-1)%K, j] = True; exclude[j,j] = True; exclude[(j+1)%K,j] = True
    #     has_vertex_inside = (inside & ~exclude).any(axis=0)   # (K,)
    #
    #     is_ear = convex & ~has_vertex_inside
    #     clip first ear, remove from idx
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

L = np.array([[0.,0.],[4.,0.],[4.,2.],[2.,2.],[2.,4.],[0.,4.]])
tris = triangulate_polygon(L)
print("triangles:", tris)
print("shape:", tris.shape)  # expect (4, 3)

def tri_area(v, t):
    A, B, C = v[t[0]], v[t[1]], v[t[2]]
    return 0.5 * abs((B-A)[0]*(C-A)[1] - (B-A)[1]*(C-A)[0])

total_area = sum(tri_area(L, t) for t in tris)
print(f"total area: {total_area:.4f}  (expected 12.0)")

In [ ]:
# ✅ Inline test suite
import numpy as np, time

def tri_area(v, t):
    A, B, C = v[t[0]], v[t[1]], v[t[2]]
    return 0.5 * abs((B-A)[0]*(C-A)[1] - (B-A)[1]*(C-A)[0])

def shoelace(v):
    x, y = v[:,0], v[:,1]
    return 0.5 * abs(np.sum(x * np.roll(y,-1) - np.roll(x,-1) * y))

# ── Test 1: trivial triangle ───────────────────────────────────────────────
tri = np.array([[0.,0.],[1.,0.],[0.,1.]])
r1 = triangulate_polygon(tri)
assert r1.shape == (1, 3), f"Shape: {r1.shape}"
assert set(r1[0]) == {0, 1, 2}, f"Indices: {r1[0]}"
print("Test 1 passed: trivial triangle")

# ── Test 2: convex square ─────────────────────────────────────────────────
sq = np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
r2 = triangulate_polygon(sq)
assert r2.shape == (2, 3), f"Shape: {r2.shape}"
total2 = sum(tri_area(sq, t) for t in r2)
assert abs(total2 - 4.0) < 1e-9, f"Area: {total2}"
print("Test 2 passed: convex square")

# ── Test 3: concave L-shape ───────────────────────────────────────────────
L = np.array([[0.,0.],[4.,0.],[4.,2.],[2.,2.],[2.,4.],[0.,4.]])
r3 = triangulate_polygon(L)
assert r3.shape == (4, 3), f"Shape: {r3.shape}"
total3 = sum(tri_area(L, t) for t in r3)
assert abs(total3 - 12.0) < 1e-9, f"Area: {total3}"
print("Test 3 passed: concave L-shape")

# ── Test 4: random 12-gon — area preserved ────────────────────────────────
rng = np.random.default_rng(17)
angles = np.sort(rng.uniform(0, 2*np.pi, 12))
v12 = np.stack([np.cos(angles), np.sin(angles)], axis=1)
r4 = triangulate_polygon(v12)
assert r4.shape == (10, 3)
total4 = sum(tri_area(v12, t) for t in r4)
assert abs(total4 - shoelace(v12)) < 1e-9, f"Area mismatch: {total4}"
print("Test 4 passed: 12-gon area preserved")

# ── Test 5: n=200, must finish < 5s ───────────────────────────────────────
rng = np.random.default_rng(31)
angles200 = np.sort(rng.uniform(0, 2*np.pi, 200))
v200 = np.stack([np.cos(angles200), np.sin(angles200)], axis=1)
t0 = time.time()
r5 = triangulate_polygon(v200)
elapsed = time.time() - t0
assert r5.shape == (198, 3), f"Shape: {r5.shape}"
total5 = sum(tri_area(v200, t) for t in r5)
assert abs(total5 - shoelace(v200)) < 1e-6, f"Area mismatch"
assert elapsed < 5.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: n=200 in {elapsed:.3f}s")

print("\nAll tests passed!")